In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

# Load your dataset — update filename to match your CSV
df = pd.read_csv('your_dataset.csv')

# Basic inspection
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()
# Check for missing/null values in each column
print("=== NULL VALUE ANALYSIS ===")
null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df) * 100).round(2)

null_report = pd.DataFrame({
    'Null Count': null_counts,
    'Null %': null_pct
}).sort_values('Null Count', ascending=False)

print(null_report[null_report['Null Count'] > 0])
print(f"\nTotal records: {len(df):,}")
print(f"Total nulls found: {null_counts.sum():,}")
# Check for duplicate transaction IDs
# Replace 'transaction_id' with the actual column name in your dataset
id_col = df.columns[0]  # assumes first column is ID — adjust if needed

print("=== DUPLICATE ID ANALYSIS ===")
duplicates = df[df.duplicated(subset=[id_col], keep=False)]
dup_count = df.duplicated(subset=[id_col]).sum()

print(f"Total duplicate IDs found: {dup_count:,}")
print(f"Duplicate rate: {dup_count/len(df)*100:.2f}%")

if dup_count > 0:
    print("\nSample duplicate records:")
    print(duplicates.head(10))
# Failure rate by merchant/transaction category
# Update column names to match your dataset
# Common names: 'category', 'merchant_category', 'type'

# Try to find category column automatically
cat_cols = [c for c in df.columns if 'cat' in c.lower()
            or 'type' in c.lower() or 'merchant' in c.lower()]
print("Possible category columns:", cat_cols)

# Use first match — adjust manually if needed
if cat_cols:
    cat_col = cat_cols[0]
    category_counts = df[cat_col].value_counts()
    print(f"\n=== TRANSACTION COUNT BY {cat_col.upper()} ===")
    print(category_counts)

    # Plot
    plt.figure(figsize=(10, 5))
    category_counts.plot(kind='bar', color='steelblue', edgecolor='white')
    plt.title('Transaction Count by Category', fontsize=14)
    plt.xlabel('Category')
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('category_chart.png', dpi=150)
    plt.show()
    print("Chart saved as category_chart.png")
    # Detect outlier transaction amounts
# Find the amount column automatically
amt_cols = [c for c in df.columns if 'amount' in c.lower()
            or 'value' in c.lower() or 'sum' in c.lower()]
print("Possible amount columns:", amt_cols)

if amt_cols:
    amt_col = amt_cols[0]
    mean = df[amt_col].mean()
    std = df[amt_col].std()
    threshold = mean + 3 * std

    outliers = df[df[amt_col] > threshold]

    print(f"\n=== OUTLIER AMOUNT ANALYSIS ===")
    print(f"Mean transaction amount: ${mean:,.2f}")
    print(f"Std deviation: ${std:,.2f}")
    print(f"Outlier threshold (mean + 3σ): ${threshold:,.2f}")
    print(f"Outlier count: {len(outliers):,}")
    print(f"Outlier rate: {len(outliers)/len(df)*100:.2f}%")
    # Final summary — print key findings
print("=" * 50)
print("  DATA QUALITY ANALYSIS — SUMMARY FINDINGS")
print("=" * 50)
print(f"\nTotal records analysed: {len(df):,}")
print(f"\nFinding 1 — Null Values:")
print(f"  {null_counts.sum():,} null values across all columns")
print(f"\nFinding 2 — Duplicate IDs:")
print(f"  {dup_count:,} duplicate transaction IDs ({dup_count/len(df)*100:.1f}%)")
print(f"\nFinding 3 — Outlier Amounts:")
if amt_cols:
    print(f"  {len(outliers):,} transactions exceed ${threshold:,.0f}")
print(f"\nRecommendation: Implement validation rules at")
print(f"ingestion to prevent these failure categories.")
print("=" * 50)
